In [31]:
#Libraries
import pandas as pd
import numpy as np
import re
from transformers import AutoTokenizer
from datasets import Dataset
import torch
import torch.nn as nn
from transformers import AutoModel
from transformers import DataCollatorWithPadding
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [32]:
#Load Dataset
train_df = pd.read_csv("/content/phishing_train.csv")
val_df = pd.read_csv("/content/phishing_validation.csv")
test_df = pd.read_csv("/content/phishing_test.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(14015, 4)
(1752, 4)
(1752, 4)


In [33]:
#Create feature extractor
URGENCY_WORDS = {
    "urgent", "urgently", "immediately", "immediate",
    "now", "today", "asap", "quickly", "deadline",
    "expire", "expired", "final", "action"
}

CREDENTIAL_WORDS = {
    "password", "passwd", "username", "login",
    "credential", "credentials", "verify", "verification",
    "authenticate", "authentication", "account"
}

THREAT_WORDS = {
    "suspend", "suspended", "terminate", "terminated",
    "blocked", "block", "close", "closed", "penalty",
    "fraud", "unauthorized", "warning", "security"
}

FINANCIAL_WORDS = {
    "payment", "pay", "invoice", "money", "bank",
    "transfer", "transaction", "refund", "credit",
    "debit", "fee", "account", "billing"
}

CTA_WORDS = {
    "click", "clicking", "visit", "open", "download",
    "confirm", "verify", "submit", "update",
    "activate", "login"
}


def count_terms(text, vocabulary):
    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    return sum(word in vocabulary for word in words)


def extract_nlp_features(text):

    text = str(text)

    words = re.findall(r"\b[a-zA-Z]+\b", text)

    word_count = max(len(words), 1)

    uppercase_words = [
        word for word in words
        if len(word) > 1 and word.isupper()
    ]

    uppercase_ratio = len(uppercase_words) / word_count

    url_count = len(
        re.findall(
            r"https?://\S+|www\.\S+",
            text,
            flags=re.IGNORECASE
        )
    )

    email_count = len(
        re.findall(
            r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
            text
        )
    )

    phone_count = len(
        re.findall(
            r"(?:\+?\d[\d\s().-]{7,}\d)",
            text
        )
    )

    features = [
        count_terms(text, URGENCY_WORDS),
        count_terms(text, CREDENTIAL_WORDS),
        count_terms(text, THREAT_WORDS),
        count_terms(text, FINANCIAL_WORDS),
        count_terms(text, CTA_WORDS),
        url_count,
        email_count,
        phone_count,
        text.count("!"),
        text.count("?"),
        uppercase_ratio,
        np.log1p(len(text))
    ]

    return np.array(features, dtype=np.float32)

In [34]:
#Generate features
train_features = np.vstack(
    train_df["processed_text"].apply(extract_nlp_features)
)

val_features = np.vstack(
    val_df["processed_text"].apply(extract_nlp_features)
)

test_features = np.vstack(
    test_df["processed_text"].apply(extract_nlp_features)
)

print("Feature shape:", train_features.shape)
print("First feature vector:")
print(train_features[0])

Feature shape: (14015, 12)
First feature vector:
[0.       0.       1.       0.       0.       0.       0.       0.
 0.       1.       0.       6.368187]


In [35]:
#Feature normalisation
feature_mean = train_features.mean(axis=0)
feature_std = train_features.std(axis=0)

feature_std[feature_std == 0] = 1.0

train_features = (
    (train_features - feature_mean) / feature_std
)

val_features = (
    (val_features - feature_mean) / feature_std
)

test_features = (
    (test_features - feature_mean) / feature_std
)

print("Feature normalisation complete.")

Feature normalisation complete.


In [36]:
#DeBERT tokenisation
MODEL_NAME = "microsoft/deberta-v3-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 512

def tokenize_function(examples):
    return tokenizer(
        examples["processed_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

In [37]:
#Convert dataset
train_dataset = Dataset.from_pandas(
    train_df[["processed_text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["processed_text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["processed_text", "label"]],
    preserve_index=False
)

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["processed_text"]
)
val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["processed_text"]
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["processed_text"]
)

Map:   0%|          | 0/14015 [00:00<?, ? examples/s]

Map:   0%|          | 0/1752 [00:00<?, ? examples/s]

Map:   0%|          | 0/1752 [00:00<?, ? examples/s]

In [52]:
#Model features define

MODEL_NAME = "microsoft/deberta-v3-large"


class GatedHybridPhishingModel(nn.Module):

    def __init__(self, model_name, num_features=12, num_labels=2):
        super().__init__()

        # Pretrained DeBERTa backbone
        self.transformer = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32
        )

        hidden_size = self.transformer.config.hidden_size

        # Project DeBERTa representation
        self.context_projection = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Project explicit NLP features
        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Learnable gate
        self.gate = nn.Sequential(
            nn.Linear(512, 256),
            nn.Sigmoid()
        )

        # Final fusion layer
        self.fusion = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Classification layer
        self.classifier = nn.Linear(256, num_labels)

        self.loss_fn = nn.CrossEntropyLoss()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        nlp_features=None,
        labels=None
    ):

        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        contextual_embedding = outputs.last_hidden_state[:, 0, :]

        context = self.context_projection(
            contextual_embedding
        )

        features = self.feature_projection(
            nlp_features.float()
        )

        gate_input = torch.cat(
            [context, features],
            dim=1
        )

        gate = self.gate(gate_input)

        fused = (
            gate * context
            + (1 - gate) * features
        )

        fused = self.fusion(fused)

        logits = self.classifier(fused)

        loss = None

        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }

In [53]:
#Model
model = GatedHybridPhishingModel(
    MODEL_NAME,
    num_features=12,
    num_labels=2
)

model.transformer.gradient_checkpointing_enable()
model.transformer.config.use_cache = False

print("Draft 2 model ready.")

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Draft 2 model ready.


In [54]:
#Check the model
nan_params = []

for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        nan_params.append(name)

print("NaN/Inf parameters:", len(nan_params))

NaN/Inf parameters: 0


In [55]:
#Set data collator
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [56]:
#Evaluating matrix
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [57]:
#Model features

training_args = TrainingArguments(
    output_dir="/content/hybrid_phishing_draft2_results",

    num_train_epochs=3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    weight_decay=0.01,

    warmup_steps=263,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=False,
    bf16=False,

    optim="adamw_torch",

    seed=42,
    data_seed=42,

    remove_unused_columns=False,

    report_to="none",
    save_total_limit=2
)

In [59]:
#Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Draft 2 Trainer ready.")

Draft 2 Trainer ready.


In [60]:
#Train the model
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.057804,0.060696,0.986872,0.990669,0.974006,0.982267
2,0.035339,0.036188,0.993721,0.998450,0.984709,0.991532
3,0.010691,0.032704,0.994863,0.996918,0.989297,0.993093


In [61]:
#Evaluate on test set
test_results = trainer.evaluate(
    test_tokenized,
    metric_key_prefix="test"
)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.010691,0.026484,3,0.993721,0.992343,0.990826,0.991584


{'test_loss': 0.026483872905373573, 'test_accuracy': 0.9937214611872146, 'test_precision': 0.9923430321592649, 'test_recall': 0.9908256880733946, 'test_f1': 0.991583779648049}


In [63]:
#Report and confusion matrix

predictions = trainer.predict(test_tokenized)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Safe Email", "Phishing Email"],
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Classification Report:
                precision    recall  f1-score   support

    Safe Email     0.9945    0.9954    0.9950      1098
Phishing Email     0.9923    0.9908    0.9916       654

      accuracy                         0.9937      1752
     macro avg     0.9934    0.9931    0.9933      1752
  weighted avg     0.9937    0.9937    0.9937      1752

Confusion Matrix:
[[1093    5]
 [   6  648]]


In [64]:
#Build the model
MODEL_NAME = "microsoft/deberta-v3-large"


class AttentionGatedHybridPhishingModel(nn.Module):

    def __init__(self, model_name, num_features=12, num_labels=2):
        super().__init__()

        self.transformer = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32
        )

        hidden_size = self.transformer.config.hidden_size

        # Attention pooling over ALL DeBERTa tokens
        self.attention_score = nn.Linear(hidden_size, 1)

        # Project contextual representation
        self.context_projection = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Project explicit NLP features
        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Gated fusion
        self.gate = nn.Sequential(
            nn.Linear(512, 256),
            nn.Sigmoid()
        )

        # Final fusion
        self.fusion = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Classifier
        self.classifier = nn.Linear(256, num_labels)

        self.loss_fn = nn.CrossEntropyLoss()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        nlp_features=None,
        labels=None
    ):

        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        token_embeddings = outputs.last_hidden_state

        # Learn attention weight for every token
        attention_scores = self.attention_score(
            token_embeddings
        ).squeeze(-1)

        # Ignore padding tokens
        attention_scores = attention_scores.masked_fill(
            attention_mask == 0,
            -1e9
        )

        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )

        # Weighted representation of the whole email
        contextual_embedding = torch.sum(
            token_embeddings * attention_weights.unsqueeze(-1),
            dim=1
        )

        # Project both branches
        context = self.context_projection(
            contextual_embedding
        )

        features = self.feature_projection(
            nlp_features.float()
        )

        # Gated fusion
        gate_input = torch.cat(
            [context, features],
            dim=1
        )

        gate = self.gate(gate_input)

        fused = (
            gate * context
            + (1 - gate) * features
        )

        fused = self.fusion(fused)

        logits = self.classifier(fused)

        loss = None

        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits
        }

In [65]:
#Check Param
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 434,475,522
Trainable parameters: 434,475,522


In [66]:
#Check
bad_params = []

for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        bad_params.append(name)

print("NaN/Inf parameters:", len(bad_params))

NaN/Inf parameters: 0


In [68]:
#Set config
training_args = TrainingArguments(
    output_dir="/content/hybrid_phishing_draft3_results",

    num_train_epochs=3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    weight_decay=0.01,

    warmup_steps=263,
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    fp16=False,
    bf16=False,

    optim="adamw_torch",

    seed=42,
    data_seed=42,

    remove_unused_columns=False,

    report_to="none",
    save_total_limit=2
)

In [70]:
#Trainer
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("Draft 3 Trainer ready.")

Draft 3 Trainer ready.


In [71]:
#Train the model
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.009183,0.057599,0.992580,0.993837,0.986239,0.990023
2,0.006512,0.085436,0.989726,0.980363,0.992355,0.986322
3,0.008514,0.062342,0.992009,0.993827,0.984709,0.989247


In [72]:
#Evaluation on test set
test_results = trainer.evaluate(
    test_tokenized,
    metric_key_prefix="test"
)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.008514,0.050852,3,0.993151,0.992331,0.989297,0.990812


{'test_loss': 0.05085229501128197, 'test_accuracy': 0.9931506849315068, 'test_precision': 0.9923312883435583, 'test_recall': 0.9892966360856269, 'test_f1': 0.9908116385911179}


In [73]:
#Report and confusion matrix
predictions = trainer.predict(test_tokenized)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Safe Email", "Phishing Email"],
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Classification Report:
                precision    recall  f1-score   support

    Safe Email     0.9936    0.9954    0.9945      1098
Phishing Email     0.9923    0.9893    0.9908       654

      accuracy                         0.9932      1752
     macro avg     0.9930    0.9924    0.9927      1752
  weighted avg     0.9931    0.9932    0.9931      1752

Confusion Matrix:
[[1093    5]
 [   7  647]]
